In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
from natsort import natsorted
import plotly.graph_objects as go
from os.path import join as pjoin
from numpy.random import MT19937, SeedSequence, RandomState

sys.path.append("../../")
import circletrack_behavior as ctb
import plotting_functions as pf

In [ ]:
## Settings
parent_dir = 'CircleTrack_Opto'
experiment_dir = 'RT_Opto1'
lin_path = f'../../../{parent_dir}/{experiment_dir}/output/lin_behav/'
circle_path = f'../../../{parent_dir}/{experiment_dir}/output/behav/'
fig_path = f'../../../{parent_dir}/{experiment_dir}/intermediate_figures'
maze_info = pd.read_csv(f'../../../{parent_dir}/{experiment_dir}/maze_yml/maze_info.csv')
chance_color = '#7d7d7d'
avg_color = 'midnightblue'
subject_color = 'darkgrey'
two_group_colors = ['midnightblue', 'darkorchid']
group_colors_dict = {'stGtACR2': 'darkorchid', 'mCherry': 'midnightblue'}
error_dict = {'stGtACR2': 'rgba(153,50,204,0.4)', 'mCherry': 'rgba(0,41,102,0.4)'}
excluded_mice = ['rto05', 'rto12']

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

In [ ]:
## Randomize port numbers for each context for each mouse
rs = RandomState(MT19937(SeedSequence(1)))
context_list = ['A', 'B', 'C']
port_list = [0, 1, 2, 3]
mouse_list = [f'rto0{x}' for x in np.arange(2, 10)] + ['rto10', 'rto11', 'rto12']
potential_combinations = [
    [['reward1', 'reward5'], ['reward2', 'reward6'], ['reward3', 'reward7'], ['reward4', 'reward8']],
    [['reward1', 'reward6'], ['reward2', 'reward5'], ['reward3', 'reward8'], ['reward4', 'reward7']],
    [['reward1', 'reward4'], ['reward2', 'reward7'], ['reward3', 'reward6'], ['reward5', 'reward8']],
    [['reward1', 'reward5'], ['reward2', 'reward6'], ['reward3', 'reward8'], ['reward4', 'reward7']],
]

output = {}
for mouse in mouse_list:
    context_ports = {'A': [], 'B': [], 'C': [], 'D': []}
    randcont = rs.randint(0, len(context_list))
    randports = rs.choice(port_list, size=4, replace=False)
    for context, ports in zip(context_list, randports):
        context_ports[context].append(potential_combinations[randcont][ports])
    output[f'{mouse}'] = context_ports
port_df = pd.DataFrame(output)

### Circle track lick accuracy and rewards.

In [ ]:
circletrack_results = {'mouse': [], 'day': [], 'sex': [], 'group': [], 'session': [], 'lick_accuracy': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass 
    else:
        mouse_path = pjoin(circle_path, mouse)
        sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
        group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
        for idx, session in enumerate(natsorted(os.listdir(mouse_path))):
            behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
            behav = behav[~behav['probe']]
            reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
            pc_thresh5 = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            circletrack_results['mouse'].append(mouse)
            circletrack_results['day'].append(idx+1)
            circletrack_results['sex'].append(sex)
            circletrack_results['group'].append(group)
            circletrack_results['session'].append(np.unique(behav['session_two'])[0])
            if session == 'rto11_10.feat':
                circletrack_results['lick_accuracy'].append(np.nan)
                circletrack_results['rewards'].append(np.nan)
            else:
                circletrack_results['lick_accuracy'].append(pc_thresh5)
                circletrack_results['rewards'].append(np.sum(behav['water']))
ct_df = pd.DataFrame(circletrack_results)
ct_df = ct_df[~pd.isna(ct_df['lick_accuracy'])]

In [ ]:
## Plot lick accuracy across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle', 'diamond'],
                                   plot_datapoints=True, x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=500)
# fig.add_vrect(x0=5.5, x1=6.5, fillcolor=chance_color, layer='below', opacity=0.5, line_width=0)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
## Plot rewards across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle', 'diamond'],
                                   plot_datapoints=True, x_title='Day', y_title='Rewards', titles=[''], height=500, width=500)
# fig.add_vrect(x0=5.5, x1=6.5, fillcolor=chance_color, layer='below', opacity=0.5, line_width=0)
fig.show()

### Look at probe accuracy.

In [ ]:
lick_dict_probe = {'mouse': [], 'experiment': [], 'sex': [], 'group': [], 'session': [], 
                   'day': [], 'num_licks': [], 'probe_acc': [], 'session_acc': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(natsorted(os.listdir(mpath))):
        behav = pd.read_feather(pjoin(mpath, session))
        if any(behav['probe']):
            behav_probe = behav[behav['probe']]
            behav_no_probe = behav[~behav['probe']]
            reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
            percent_correct = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            session_pc = ctb.lick_accuracy(behav_no_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            lick_dict_probe['mouse'].append(mouse)
            lick_dict_probe['experiment'].append(behav['cohort'].unique()[0])
            lick_dict_probe['sex'].append(sex)
            lick_dict_probe['group'].append(group)
            lick_dict_probe['day'].append(idx+1)
            lick_dict_probe['session'].append(np.unique(behav['session_two'])[0])
            if session == 'rto11_10.feat':
                lick_dict_probe['num_licks'].append(np.nan)
                lick_dict_probe['probe_acc'].append(np.nan)
                lick_dict_probe['session_acc'].append(np.nan)
                lick_dict_probe['rewards'].append(np.nan)
            else:
                lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                lick_dict_probe['probe_acc'].append(percent_correct)
                lick_dict_probe['session_acc'].append(session_pc)
                lick_dict_probe['rewards'].append(np.sum(behav_no_probe['water']))
        else:
            pass
probe_df = pd.DataFrame(lick_dict_probe)
probe_df = probe_df[~pd.isna(probe_df['probe_acc'])]

In [ ]:
## Plot probe performance for first and last day in A
avg_probe = probe_df.groupby(['day', 'group'], as_index=False).agg({'probe_acc': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', titles=[''])

for group in ['stGtACR2', 'mCherry']:
    gdata = avg_probe[avg_probe['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['probe_acc']['mean'], mode='markers', marker_color=group_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=1.5, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['probe_acc']['sem'], thickness=2.5)))
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(range=[0, 100])
fig.show()